In [1]:
from dataclasses import dataclass
from enum import Enum
from hashlib import sha256

# Blockchain

In [2]:
class EventLog:
    listeners = []

    def register(listener):
        EventLog.listeners.append(listener)

    def emit(event):
        for listener in EventLog.listeners:
            listener.onEvent(event)

In [3]:
class Contract:
    def emitEvent(self, event):
        EventLog.emit((self, event))

@dataclass
class Address:
    value: str

    def __str__(self):
        return self.value

    def __hash__(self):
        return hash(self.value)

In [ ]:
class EncryptedType(Enum):
    EBOOL = 0
    EUINT = 1

class OpType(Enum):
    CONSTANT = 0
    ADD = 1
    SUB = 2
    EQ = 3
    LT = 4
    LEQ = 5
    GT = 6
    GEQ = 7
    SELECT = 8

In [ ]:
@dataclass
class Handle:
    raw: str

    def encryptedType(self) -> EncryptedType:
        handleTypeCode = self.raw[0:2]
        if handleTypeCode == '00':
            return EncryptedType.EBOOL
        elif handleTypeCode == '01':
            return EncryptedType.EUINT
        else:
            raise Exception(f"Unknown handle type code: {handleTypeCode}") 

    def fromRaw(raw: str) -> Handle:
        return Handle(raw)

    def fromComponents(prehandle: str, typeCode: EncryptedType) -> Handle:
        handle = typeCode.value.to_bytes().hex() + prehandle[2:]
        return Handle(handle)

    def newConstantHandle(value: int, returnType: FheType) -> Handle:
        preimage = str(value) + '|' + returnType.name
        prehandle = sha256(preimage.encode()).hexdigest()
        handle = returnType.value.to_bytes().hex() + prehandle[2:]
        return Handle.fromComponents(prehandle, returnType)

    def newComputeHandle(opType: OpType, returnType: EncryptedType, *handles) -> Handle:
        preimage = ''.join(handle.raw for handle in handles) + opType.name
        prehandle = sha256(preimage.encode()).hexdigest()
        handle = returnType.value.to_bytes().hex() + prehandle[2:]
        return Handle.fromComponents(prehandle, returnType)

    def __str__(self):
        return f'..{self.raw[:10]}'

    def __hash__(self):
        return hash(self.raw)

In [5]:
@dataclass
class ExternalEuint:
    ciphertext: int

@dataclass
class ExternalEbool:
    ciphertext: int

In [ ]:
@dataclass
class SymbolicEuint:
    handle: Handle

@dataclass
class SymbolicEbool:
    handle: Handle

class Op:
    def __str__(self):
        return f"{self.__class__.__name__}({', '.join(f'{k}={v}' for k, v in self.__dict__.items())})"

@dataclass
class ConstantOp(Op):
    value: int
    result: Handle

@dataclass
class IngestOp(Op):
    ciphertext: int
    result: Handle

@dataclass
class AddOp(Op):
    lhs: Handle
    rhs: Handle
    result: Handle

@dataclass
class SubOp(Op):
    lhs: Handle
    rhs: Handle
    result: Handle

@dataclass
class LeqOp(Op):
    lhs: Handle
    rhs: Handle
    result: Handle

@dataclass
class SelectOp(Op):
    selector: Handle
    trueValue: Handle
    falseValue: Handle
    result: Handle

class TrustedExecutor(Contract):
    acl: TrustedAcl

    def __init__(self, acl: TrustedAcl):
        self.acl = acl

    def _newExternalHandle(self, returnType: EncryptedType, ciphertext: int) -> Handle:
        preimage = str(ciphertext)
        prehandle = sha256(preimage.encode()).hexdigest()
        handle = returnType.value.to_bytes().hex() + prehandle[2:]
        return Handle.fromComponents(prehandle, returnType)

    def zero(self) -> SymbolicEuint:
        return self.constantEuint(0)

    def ingestEbool(self, external: ExternalEbool) -> SymbolicEbool:
        ciphertext = external.ciphertext
        result_handle = self._newExternalHandle(EncryptedType.EBOOL, ciphertext)
        self.emitEvent(IngestOp(ciphertext, result_handle))
        return SymbolicEbool(result_handle)

    def ingestEuint(self, external: ExternalEuint) -> SymbolicEuint:
        ciphertext = external.ciphertext
        result_handle = self._newExternalHandle(EncryptedType.EUINT, ciphertext)
        self.emitEvent(IngestOp(ciphertext, result_handle))
        return SymbolicEuint(result_handle)

    def constantEuint(self, value: int) -> SymbolicEuint:
        result_handle = self._newConstantHandle(value, EncryptedType.EUINT)
        self.emitEvent(ConstantOp(value, result_handle))
        return SymbolicEuint(result_handle)

    def constantEbool(self, value: bool) -> SymbolicEbool:
        encoded_value = 1 if value else 0
        result_handle = self._newConstantHandle(encoded_value, EncryptedType.EBOOL)
        self.emitEvent(ConstantOp(encoded_value, result_handle))
        return SymbolicEbool(handle)

    def add(self, x: SymbolicEuint, y: SymbolicEuint) -> SymbolicEuint:
        assert x.handle.encryptedType() == EncryptedType.EUINT
        assert y.handle.encryptedType() == EncryptedType.EUINT
        result_handle = Handle.newComputeHandle(OpType.ADD, EncryptedType.EUINT, x.handle, y.handle)
        self.emitEvent(AddOp(x.handle, y.handle, result_handle))
        return SymbolicEuint(result_handle)

    def sub(self, x: SymbolicEuint, y: SymbolicEuint) -> SymbolicEuint:
        assert x.handle.encryptedType() == EncryptedType.EUINT
        assert y.handle.encryptedType() == EncryptedType.EUINT
        result_handle = Handle.newComputeHandle(OpType.SUB, EncryptedType.EUINT, x.handle, y.handle)
        self.emitEvent(SubOp(x.handle, y.handle, result_handle))
        return SymbolicEuint(result_handle)

    def leq(self, x: SymbolicEuint, y: SymbolicEuint) -> SymbolicEbool:
        assert x.handle.encryptedType() == EncryptedType.EUINT
        assert y.handle.encryptedType() == EncryptedType.EUINT
        result_handle = Handle.newComputeHandle(OpType.LEQ, EncryptedType.EBOOL, x.handle, y.handle)
        self.emitEvent(LeqOp(x.handle, y.handle, result_handle))
        return SymbolicEbool(result_handle)

    def select(self, selector: SymbolicEbool, trueValue: SymbolicEuint, falseValue: SymbolicEuint) -> SymbolicEuint:
        assert selector.handle.encryptedType() == EncryptedType.EBOOL
        assert true.handle.encryptedType() == EncryptedType.EUINT
        assert false.handle.encryptedType() == EncryptedType.EUINT
        result_handle = Handle.newComputeHandle(OpType.SELECT, EncryptedType.EUINT, selector.handle, true.handle, false.handle)
        self.emitEvent(SelectOp(selector.handle, true.handle, false.handle, result_handle))
        return SymbolicEuint(result_handle)

In [7]:
class TrustedAcl(Contract):

    computeAllowance: set()

    def allowCompute(self, handle: Handle, address: Address):
        self.computeAllowance[handle].add(address)
    
    def allowPublicDecryption(self, handle, address):
        self.decryptAllowance[handle].add(address)

    def allowUserDecryption(self, handle, address):
        self.publicDecryptAllowance[handle].add(address)

# Plaintext Coprocessor

In [ ]:
class PlaintextCoprocessor:
    computeGraph: list[Op]
    concreteValues: dict[Handle, Union[int, bool]]

    def __init__(self):
        self.computeGraph = []
        self.concreteValues = {}
        EventLog.register(self)

    def registerTrustedExecutor(executor: TrustedExecutor):
        self.executor = executor

    def reset(self):
        self.computeGraph = []
        self.concreteValues = {}

    def onEvent(self, event):
        (contract, op) = event
        if self.executor and contract == self.executor:
            self.computeGraph.append(op)

    def printComputeGraph(self):
        for op in self.computeGraph:
            print(op)

    def printConcreteValues(self):
        for key, value in coprocessor.concreteValues.items():
            print(f"{key}: {value}")

    def compute(self):
        for op in self.computeGraph:
            self.computeOp(op)

    def computeOp(self, op):
        if isinstance(op, ConstantOp):
            if op.result.typeCode() == EncryptedType.EBOOL:
                self.concreteValues[op.result] = op.value == 0
            elif op.result.typeCode() == EncryptedType.EUINT:
                self.concreteValues[op.result] = op.value
            
        elif isinstance(op, IngestOp):
            if op.result.typeCode() == EncryptedType.EBOOL:
                self.concreteValues[op.result] = ConcreteEbool(False) if op.ciphertext == 0 else ConcreteEbool(True)
            elif op.result.typeCode() == EncryptedType.EUINT:
                self.concreteValues[op.result] = ConcreteEuint(op.ciphertext)

        elif isinstance(op, AddOp):
            lhs = self.concreteValues.get(op.lhs); assert lhs is not None
            rhs = self.concreteValues.get(op.rhs); assert rhs is not None
            self.concreteValues[op.result] = lhs + rhs

        elif isinstance(op, SubOp):
            lhs = self.concreteValues.get(op.lhs); assert lhs is not None
            rhs = self.concreteValues.get(op.rhs); assert rhs is not None
            self.concreteValues[op.result] = lhs - rhs

        elif isinstance(op, LeqOp):
            lhs = self.concreteValues.get(op.lhs); assert lhs is not None
            rhs = self.concreteValues.get(op.rhs); assert rhs is not None
            self.concreteValues[op.result] = lhs.raw <= rhs.raw

        elif isinstance(op, SelectOp):
            selector = self.concreteValues.get(op.selector); assert selector is not None
            trueValue = self.concreteValues.get(op.trueValue); assert trueValue is not None
            falseValue = self.concreteValues.get(op.falseValue); assert falseValue is not None
            self.concreteValues[op.result] = trueValue if selector else falseValue
            
        else:
            raise Exception(f"Unknown op event type: {type(op)}")

# FHE Coprocessor

In [23]:
@dataclass
class Ciphertext:
    fheType: FheType
    plaintext: int

class TFHE:

    DIVISION_ERROR = 2^32 - 1
    TRUE = 1
    FALSE = 0
    
    def trivialUint(x: int) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x)

    def trivialBool(b: bool) -> Ciphertext:
        return Ciphertext(FheType.EBOOL, TRUE if b else FALSE)

    def fheAdd(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext + y.plaintext)

    def fheSub(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext - y.plaintext)

    def fheMul(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext * y.plaintext)

    def fheDiv(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        # Note that division returns an encrypted constant instead of failing on y == 0
        if y.plaintext == 0:
            return Ciphertext(FheType.EUINT, DIVISION_ERROR)
        return Ciphertext(FheType.EUINT, x.plaintext / y.plaintext)

    def fheEq(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext == y.plaintext)

    def fheLeq(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext <= y.plaintext)

    def fheLt(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext < y.plaintext)

    def fheGeq(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext >= y.plaintext)

    def fheGt(x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext > y.plaintext)

    def fheAnd(x: Ciphertext, y: Ciphertext) -> BoolCiphertext:
        return Ciphertext(FheType.EBOOL, TRUE if x.plaintext != FALSE and y.plaintext != FALSE else FALSE)

    def fheOr(x: Ciphertext, y: Ciphertext) -> BoolCiphertext:
        return Ciphertext(FheType.EBOOL, TRUE if x.plaintext != FALSE or y.plaintext != FALSE else FALSE)

    def fheXor(x: Ciphertext, y: Ciphertext) -> BoolCiphertext:
        return Ciphertext(FheType.EBOOL, TRUE if x.plaintext != y.plaintext else FALSE)

    def fheSelect(c: BoolCiphertext, x: Ciphertext, y: Ciphertext) -> Ciphertext:
        return Ciphertext(FheType.EUINT, x.plaintext if c.plaintext == TRUE else y.plaintext)

fhe = TFHE()
#decryptionKey, evaluationKey = fhe.generateKeyset() # TODO how do we model that the KMS does this?

In [ ]:
@dataclass
class ConcreteEuint:
    ciphertext: UintCiphertext

@dataclass
class ConcreteEbool:
    ciphertext: BoolCiphertext

class FheCoprocessor:
    computeGraph: list[Op]
    concreteValues: dict[Handle, Union[ConcreteEuint, ConcreteEbool]]
    ciphertextStorage: CiphertextStorage

    def __init__(self, ciphertextStorage: CiphertextStorage):
        self.ciphertextStorage = ciphertextStorage
        self.computeGraph = []
        self.concreteValues = {}
        EventLog.register(self)

    def registerTrustedExecutor(executor: TrustedExecutor):
        self.executor = executor

    def reset(self):
        self.computeGraph = []
        self.concreteValues = {}

    def onEvent(self, event):
        (contract, op) = event
        if self.executor and contract == self.executor:
            self.computeGraph.append(op)

    def printComputeGraph(self):
        for op in self.computeGraph:
            print(op)

    def printConcreteValues(self):
        for key, value in coprocessor.concreteValues.items():
            print(f"{key}: {value}")

    def compute(self):
        for op in self.computeGraph:
            self.computeOp(op)

    def computeOp(self, op):
        if isinstance(op, ConstantOp):
            if op.result.typeCode() == FheType.EBOOL:
                self.concreteValues[op.result] = ConcreteEbool(False) if op.constant == 0 else ConcreteEbool(True)
            elif op.result.typeCode() == FheType.EUINT:
                self.concreteValues[op.result] = ConcreteEuint(op.value)
            
        elif isinstance(op, IngestOp):
            if op.result.typeCode() == FheType.EBOOL:
                self.concreteValues[op.result] = ConcreteEbool(False) if op.ciphertext == 0 else ConcreteEbool(True)
            elif op.result.typeCode() == FheType.EUINT:
                self.concreteValues[op.result] = ConcreteEuint(op.ciphertext)

        elif isinstance(op, AddOp):
            lhs = self.concreteValues.get(op.lhs)
            rhs = self.concreteValues.get(op.rhs)
            if lhs is None or rhs is None:
                raise Exception(f"Missing concrete value for lhs or rhs in AddOp: {op}")
            result = lhs.raw + rhs.raw
            self.concreteValues[op.result] = ConcreteEuint(result)

        elif isinstance(op, SubOp):
            lhs = self.concreteValues.get(op.lhs)
            rhs = self.concreteValues.get(op.rhs)
            if lhs is None or rhs is None:
                raise Exception(f"Missing concrete value for lhs or rhs in SubOp: {op}")
            result = lhs.raw - rhs.raw
            self.concreteValues[op.result] = ConcreteEuint(result)

        elif isinstance(op, LeqOp):
            lhs = self.concreteValues.get(op.lhs)
            rhs = self.concreteValues.get(op.rhs)
            if lhs is None or rhs is None:
                raise Exception(f"Missing concrete value for lhs or rhs in LeqOp: {op}")
            result = lhs.raw <= rhs.raw
            self.concreteValues[op.result] = ConcreteEbool(result)

        elif isinstance(op, SelectOp):
            selector = self.concreteValues.get(op.selector)
            trueValue = self.concreteValues.get(op.trueValue)
            falseValue = self.concreteValues.get(op.falseValue)
            if selector is None or trueValue is None or falseValue is None:
                raise Exception(f"Missing concrete value for selector, trueValue, or falseValue in SelectOp: {op}")
            result = trueValue.raw if selector.raw else falseValue.raw
            self.concreteValues[op.result] = ConcreteEuint(result)
            
        else:
            raise Exception(f"Unknown op event type: {type(op)}")

In [ ]:
class PheCoprocessor:
    pass

confidentialProtocol = PheCoprocessor + PheKMS

In [ ]:
class MpcCoprocessor:
    pass

confidentialProtocol = MpcCoprocessor

In [ ]:
class EnclaveCoprocessor:
    pass

confidentialProtocol = EnclaveCoprocessor

In [36]:
class MPC:
    
    def thresholdKeyGeneration():
        pass

    def thresholdUserDecryption(ciphertext, userEncryptionKey):
        pass

    def thresholdPublicDecryption(ciphertext):
        pass


In [48]:
class KMS:
    acl: TrustedAcl
    ciphertextStorage: CiphertextStorage

    def __init__(self, ciphertextStorage: CiphertextStorage):
        self.ciphertextStorage = ciphertextStorage

    def registerTrustedAcl(self, acl: TrustedAcl):
        self.acl = acl

    def publicDecryption(self, handle: Handle, caller: Address) -> int:
        if self.acl.isAllowed(handle, caller):
            return self.ciphertextStorage.fetchCiphertext(handle)

    def padDecryption(self):
        #TODO
        pass

In [49]:
class ERC20:
    balances: dict[Address, SymbolicEuint]

    def __init__(self, executor: TrustedExecutor):
        self.executor = executor
        self.balances = {}

    def getBalance(self, address: Address) -> SymbolicEuint:
        return self.balances.get(address, self.executor.zero())

    def mint(self, address: Address, amount: int):
        self._mint(address, self.executor.constantEuint(amount))

    def transfer(self, fromAddress: Address, toAddress: Address, amount: int):
        self._transfer(fromAddress, toAddress, self.executor.constantEuint(amount))

    def confidentialMint(self, address: Address, amount: ExternalEuint):
        self._mint(address, self.executor.ingestEuint(amount))

    def confidentialTransfer(self, fromAddress: Address, toAddress: Address, amount: ExternalEuint):
        self._transfer(fromAddress, toAddress, self.executor.ingestEuint(amount))

    def _mint(self, address: Address, amount: SymbolicEuint):
        balance = self.balances.get(address, self.executor.zero())
        self.balances[address] = self.executor.add(balance, amount)

    def _transfer(self, fromAddress: Address, toAddress: Address, amount: SymbolicEuint):
        fromBalance = self.balances.get(fromAddress, self.executor.zero())
        sufficient = self.executor.leq(amount, fromBalance)
        adjustedAmount = self.executor.select(sufficient, amount, self.executor.zero())
        self.balances[fromAddress] = self.executor.sub(fromBalance, adjustedAmount)
        toBalance = self.balances.get(toAddress, self.executor.zero())
        self.balances[toAddress] = self.executor.add(toBalance, adjustedAmount)

In [50]:
acl = TrustedAcl()
executor = TrustedExecutor(acl)

In [51]:
class CiphertextStorage:
    storage = {}

    def storeCiphertext(self, handle: Handle, ciphertext: Ciphertext):
        existingCiphertext = self.storage.get(handle)
        assert existingCiphertext is None or existingCiphertext == ciphertext
        self.storage[handle] = ciphertext

    def fetchCiphertext(self, handle: Handle) -> Ciphertext:
        return self.get(handle)


In [53]:
ciphertextStorage = CiphertextStorage()

kms = KMS(ciphertextStorage)
#kms.generateKeyset() # TODO trigger via Ethereum governance?
kms.registerTrustedAcl(acl)

coprocessor = Coprocessor(ciphertextStorage)
coprocessor.registerTrustedExecutor(executor)

TypeError: Coprocessor.__init__() takes 1 positional argument but 2 were given

In [17]:
alice = Address("0xAlice")
bob = Address("0xBob")

In [ ]:
token = ERC20(executor)

In [ ]:
token.mint(alice, 100)

coprocessor.compute()

print(f"Alice's balance: {kms.publicDecryption(token.getBalance(alice).handle)}")
print(f"Bob's balance: {kms.publicDecryption(token.getBalance(bob).handle)}")

In [ ]:
token.transfer(alice, bob, 25)

coprocessor.compute()

print(f"Alice's balance: {kms.publicDecryption(token.getBalance(alice).handle)}")
print(f"Bob's balance: {kms.publicDecryption(token.getBalance(bob).handle)}")

In [ ]:
token.confidentialTransfer(alice, bob, ExternalEuint(25))

coprocessor.compute()

print(f"Alice's balance: {kms.publicDecryption(token.getBalance(alice).handle)}")
print(f"Bob's balance: {kms.publicDecryption(token.getBalance(bob).handle)}")